# 02 - Price analysis (EDA)

How asking prices vary by property type and location, and which attributes are associated with price.

**Inputs:** `data/processed/listings.parquet`, `data/processed/priced_listings.parquet` (written by notebook 01)
**Outputs:** figures and tables only (no data files)
**Previous:** `01_eda_data-quality.ipynb`

**Research objectives**

Purpose: understand how asking prices of Vietnamese property listings vary by location and property type, and how reliable the data is for those comparisons.

1. **Data quality:** How complete, duplicated and valid are the listings, and which rows are kept for price analysis?
2. **Price patterns:** How do asking price and price per m² differ by province, district and property type among listings with disclosed price and area?
3. **Associated factors:** Which attributes (area, frontage, floors, bedrooms, bathrooms) are most associated with asking price and price per m²? (Spearman on raw values, Pearson on log10.)

Assumption: every listing is treated as a property for sale, and `price` is the price the seller asks for. Results only cover listings that show a price.


## 0. SET UP

In [ ]:
import sys

# Notebooks live in notebooks/, so the repo root is one level up.
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from eda_utils.config import RANDOM_STATE
from eda_utils.helpers import (
    format_price_table,
    pairwise_corr,
    plot_box,
    price_summary,
    title_with_n,
)

np.random.seed(RANDOM_STATE)

pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12

In [ ]:
df = pd.read_parquet("../data/processed/listings.parquet")
df_priced = pd.read_parquet("../data/processed/priced_listings.parquet")
n_priced = len(df_priced)

numeric_cols = [
    "price", "area", "floor_count", "frontage_width", "house_depth",
    "road_width", "bedroom_count", "bathroom_count",
]
print("df :", df.shape)
print("df_priced:", df_priced.shape)

## 2. EXPLORATORY DATA ANALYSIS

All listings are treated as for sale. Price-specific figures use `df_priced` (positive price and area) unless noted. Group statistics require **n ≥ 100**.


### 2.1. General dataset profile


In [ ]:
display(df[numeric_cols].describe())
print(title_with_n("Numeric profile", "df", len(df)))

### 2.2. Price distributions


In [ ]:
fig, axes = plt.subplots(2, 2)
panels = [
    (axes[0][0], "price", False, "Price (VND)", "steelblue"),
    (axes[0][1], "price", True, "Price log10(VND)", "darkorange"),
    (axes[1][0], "area", False, "Area (m2)", "seagreen"),
    (axes[1][1], "area", True, "Area log10(m2)", "darkorange"),
]
for ax, col, log_x, title, color in panels:
    values = np.log10(df_priced[col]) if log_x else df_priced[col]
    ax.hist(values, bins=60, color=color)
    ax.set_title(title_with_n(title, "df_priced", len(df_priced)))
    ax.set_xlabel(f"log10({col})" if log_x else col)

plt.tight_layout()
plt.show()

### 2.3. Property type and Geographical analysis

Medians of `df_priced` (prices are skewed). Price is shown in billion VND and price per m² in million VND. Groups need n ≥ 100.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, top_n, title in [
    (axes[0], "property_type_name", 8, "Property type share"),
    (axes[1], "province_name", 8, "Province share"),
]:
    share = df[col].value_counts(normalize=True, dropna=False) * 100
    top = share.head(top_n)
    if len(share) > top_n:
        top = pd.concat([top, pd.Series({"Other": share.iloc[top_n:].sum()})])
    top = top.iloc[::-1]

    ax.barh(top.index.astype(str), top.values, color="steelblue")
    for y, v in enumerate(top.values):
        ax.text(v, y, f" {v:.1f}%", va="center", fontsize=9)
    ax.set_xlabel("% of listings")
    ax.set_xlim(0, top.values.max() * 1.15)
    ax.set_title(title_with_n(title, "df", len(df)))

plt.tight_layout()
plt.show()

In [ ]:
type_summary = price_summary(df_priced, "property_type_name")
prov_summary = price_summary(df_priced, "province_name")

print(title_with_n("Median price and price per m² by property type", "df_priced", len(df_priced)))
display(format_price_table(type_summary))
print(title_with_n("Median price and price per m² by province (top 15 by n)", "df_priced", len(df_priced)))
display(format_price_table(prov_summary.head(15)))

In [ ]:
dist_summary = price_summary(df_priced, ["province_name", "district_name"])

print(title_with_n("Median price and price per m² by district (top 15 by n)", "df_priced", len(df_priced)))
display(format_price_table(dist_summary.head(15)))

In [ ]:
top_prov = df_priced["province_name"].value_counts().head(10).index
type_order = df_priced["property_type_name"].value_counts().index
sub = df_priced[df_priced["province_name"].isin(top_prov)]
cell_stats = (
    sub.groupby(["province_name", "property_type_name"], observed=True)["price_per_m2"]
    .agg(n="size", median="median")
    .reset_index()
)
cell_stats = cell_stats[cell_stats["n"] >= 100]
heat = cell_stats.pivot(index="province_name", columns="property_type_name", values="median") / 1e6
heat = heat.reindex(index=top_prov, columns=type_order)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "million VND / m²"}, ax=ax)
ax.set_title(title_with_n(
    "Median price_per_m2 by province × property type (n≥100 cells)",
    "df_priced",
    len(df_priced),
))
plt.tight_layout()
plt.show()

### 2.4. Area–price relationships


In [ ]:
scatter_n = min(20_000, len(df_priced))
scatter_df = df_priced.sample(n=scatter_n, random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scatter_df["area"], scatter_df["price"], s=6, alpha=0.25, c="steelblue")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("area (m²)")
ax.set_ylabel("price (VND)")
ax.set_title(title_with_n(f"Area vs price (sample {scatter_n:,})", "df_priced", len(df_priced)))
plt.tight_layout()
plt.show()


In [ ]:
plot_box(df_priced, "property_type_name", "price", type_order,
         "price (VND)", "Asking price by property type")

### 2.5. Price per square meter analysis


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log10(df_priced["price_per_m2"]), bins=60, color="purple")
ax.set_title(title_with_n("price_per_m2 log10 scale", "df_priced", len(df_priced)))
ax.set_xlabel("log10(price_per_m2)")
plt.tight_layout()
plt.show()

In [ ]:
plot_box(df_priced, "province_name", "price_per_m2", top_prov,
         "price_per_m2 (VND/m²)", "price_per_m2 by province (top 10 by n)")

In [ ]:
plot_box(df_priced, "property_type_name", "price_per_m2", type_order,
         "price_per_m2 (VND/m²)", "price_per_m2 by property type")

### 2.6. Correlation analysis

Spearman on raw values, and Pearson on log10-transformed price, area and price_per_m2. Pairwise deletion on `df_priced`; pairs with n < 1,000 are not reported.

In [ ]:
CORR_COLS = [
    "price", "price_per_m2", "area", "floor_count", "frontage_width",
    "house_depth", "road_width", "bedroom_count", "bathroom_count",
]
spearman1, n1 = pairwise_corr(df_priced, CORR_COLS, method="spearman")

print(title_with_n("Pairwise n", "df_priced", len(df_priced)))
display(n1.astype(int))
print("Pairs with n < 1000 are NaN (not reported) in correlation matrices.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(spearman1.astype(float), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title(title_with_n("Spearman", "df_priced", len(df_priced)))
plt.tight_layout()
plt.show()

In [ ]:
log_frame = df_priced.copy()
for c in ["price", "area", "price_per_m2"]:
    log_frame[f"log10_{c}"] = np.log10(log_frame[c].where(log_frame[c] > 0))
log_cols = ["log10_price", "log10_area", "log10_price_per_m2"] + [
    c for c in CORR_COLS if c not in {"price", "area", "price_per_m2"}
]
pearson_log, _ = pairwise_corr(log_frame, log_cols, method="pearson")
print(title_with_n("Pearson with log10(price, area, price_per_m2)", "df_priced", len(df_priced)))
display(pearson_log)


### 2.7. Key findings and summary

Wording is associational; prices are asking prices.

In [ ]:
top_prov_row = prov_summary.loc[prov_summary["median_price_per_m2"].idxmax()]
bot_prov_row = prov_summary.loc[prov_summary["median_price_per_m2"].idxmin()]
top_type_row = type_summary.loc[type_summary["median_price_per_m2"].idxmax()]

sp_area = spearman1.loc["area", "price"]
sp_n = int(n1.loc["area", "price"])

med_price = float(df_priced["price"].median())
med_ppm2 = float(df_priced["price_per_m2"].median())

insights = [
    f"1. Among df_priced (n={n_priced:,}), median asking price is {med_price:,.0f} VND and median asking price_per_m2 is {med_ppm2:,.0f} VND/m².",
    f"2. Among provinces with n≥100, median asking price_per_m2 is highest in {top_prov_row['province_name']} ({top_prov_row['median_price_per_m2']:,.0f} VND/m², n={int(top_prov_row['n']):,}) and lowest in {bot_prov_row['province_name']} ({bot_prov_row['median_price_per_m2']:,.0f} VND/m², n={int(bot_prov_row['n']):,}).",
    f"3. Among property types with n≥100, median asking price_per_m2 is highest for {top_type_row['property_type_name']} ({top_type_row['median_price_per_m2']:,.0f} VND/m², n={int(top_type_row['n']):,}).",
    f"4. Area is associated with asking price (Spearman={sp_area:.3f}, pairwise n={sp_n:,}) on df_priced.",
]
for line in insights:
    print(line)